# 🧠 Phase 3: Semantic Segmentation Using U-Net Architecture
In this notebook, we ingest the 624 preprocessed temporal SAR patches, build a deep convolutional U-Net, and train it on a Kaggle GPU accelerator to segment flooded vs. non-flooded pixels.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Kaggle dynamic directory finder
base_input_dir = '/kaggle/input'

# Dhoondhte hain ki exact data folder kahan par hai
found_dir = None
for root, dirs, files in os.walk(base_input_dir):
    if 'images' in dirs and 'masks' in dirs:
        found_dir = root
        break

if found_dir:
    images_dir = os.path.join(found_dir, 'images')
    masks_dir = os.path.join(found_dir, 'masks')
    
    images_list = sorted(os.listdir(images_dir))
    masks_list = sorted(os.listdir(masks_dir))
    
    print("✅ DATA CONNECTION SUCCESSFUL BHAI!")
    print(f"📂 Data Base Path: {found_dir}")
    print(f"📦 Total Images patches found: {len(images_list)}")
    print(f"📦 Total Masks patches found: {len(masks_list)}")
else:
    print("❌ Abhi bhi nahi mila! Ek baar hum check karenge.")

# GPU verification status
print(f"🔥 Active GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

### 🔄 Step 2: Data Ingestion and Train-Validation Splitting
We load all 624 NumPy patch matrices from disk into system RAM, ensuring correct float32 dtype mapping, and execute a deterministic 80/20 stratified split for robust model evaluation.

In [ ]:
X_data = []
Y_data = []

print("⏳ Loading arrays into memory safely...")
for img_name, mask_name in zip(images_list, masks_list):
    img_array = np.load(os.path.join(images_dir, img_name))
    mask_array = np.load(os.path.join(masks_dir, mask_name))
    
    X_data.append(img_array)
    Y_data.append(mask_array)

# NumPy arrays mein convert karo
X_data = np.array(X_data, dtype=np.float32)
Y_data = np.array(Y_data, dtype=np.float32)

# --- CRITICAL FIX 1: Clean EVERYTHING (Images AND Masks) ---
print("🧹 Cleaning NaNs from both Images and Masks...")
X_data = np.nan_to_num(X_data, nan=0.0, posinf=0.0, neginf=0.0)
Y_data = np.nan_to_num(Y_data, nan=0.0, posinf=0.0, neginf=0.0)

# --- CRITICAL FIX 2: Safe Min-Max Normalization for Images ---
min_val = np.min(X_data)
max_val = np.max(X_data)
if max_val - min_val > 0:
    X_data = (X_data - min_val) / (max_val - min_val)

# --- CRITICAL FIX 3: Force Dimensions for Masks (624, 256, 256, 1) ---
if len(Y_data.shape) == 3:
    Y_data = np.expand_dims(Y_data, axis=-1)

# Ensure masks are strict binary (0 or 1)
Y_data = np.where(Y_data > 0.5, 1.0, 0.0).astype(np.float32)

# Split data now
X_train, X_val, y_train, y_val = train_test_split(X_data, Y_data, test_size=0.2, random_state=42)

print("\n📊 --- Final Verify Shape and Ranges ---")
print(f"💪 X_train -> Shape: {X_train.shape} | Min: {X_train.min()} | Max: {X_train.max()}")
print(f"💪 y_train -> Shape: {y_train.shape} | Unique values: {np.unique(y_train)}")
print(f"🧪 X_val   -> Shape: {X_val.shape} | Min: {X_val.min()} | Max: {X_val.max()}")
print(f"🧪 y_val   -> Shape: {y_val.shape} | Unique values: {np.unique(y_val)}")

In [ ]:
# 1. Clean NaN values from train and validation sets
print("🧹 Cleaning hidden NaN values from data...")
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)

# 2. Robust Min-Max Normalization Function using nanmin/nanmax safely
def robust_normalize(data):
    min_val = np.min(data)
    max_val = np.max(data)
    if max_val - min_val == 0:
        return data
    return (data - min_val) / (max_val - min_val)

# 3. Apply normalization
X_train = robust_normalize(X_train)
X_val = robust_normalize(X_val)

print("✨ BOOM! Data completely clean aur normalize ho gaya bhai!")
print(f"📊 New X_train Range -> Min: {X_train.min()} | Max: {X_train.max()}")

### 🧬 Step 3: Defining the Legendary U-Net Architecture
We construct a classic U-Net architecture consisting of an **Encoder (Contracting Path)** to extract multi-scale spatial temporal SAR features, followed by a **Decoder (Expanding Path)** with skip-connections to reconstruct high-resolution binary flood prediction masks.

In [ ]:
# Force expand dimensions of masks just to be 100% structurally safe for Keras

if len(y_train.shape) == 3:
    y_train = np.expand_dims(y_train, axis=-1)
if len(y_val.shape) == 3:
    y_val = np.expand_dims(y_val, axis=-1)

def build_unet(input_shape=(256, 256, 2)):
    inputs = layers.Input(input_shape)
    
    # --- ENCODER (Downsampling) ---
    # Block 01
    c1 = layers.Conv2D(32, (3, 3), activation = 'relu', padding = 'same')(inputs)
    c1 = layers.Conv2D(32, (3, 3), activation = 'relu', padding = 'same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)

    # Block 02
    c2 = layers.Conv2D(64, (3, 3), activation = 'relu', padding = 'same')(p1)
    c2 = layers.Conv2D(64, (3, 3), activation = 'relu', padding = 'same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)

    # Block 03
    c3 = layers.Conv2D(128, (3, 3), activation = 'relu', padding = 'same')(p2)
    c3 = layers.Conv2D(128, (3, 3), activation = 'relu', padding = 'same')(c3)

    # --- DECODER (Upsampling + Skip Connections) ---
    # Block 4 (Up-conv 2)
    u4 = layers.Conv2DTranspose(64, (2, 2), strides = (2, 2), padding = 'same')(c3)
    u4 = layers.concatenate([u4, c2])
    c4 = layers.Conv2D(64, (3, 3), activation = 'relu', padding = 'same')(u4)
    c4 = layers.Conv2D(64, (3, 3), activation = 'relu', padding = 'same')(c4)

    # Block 05  (up-conv 1)
    u5 = layers.Conv2DTranspose(32, (2, 2), strides = (2, 2), padding = 'same')(c4)
    u5 = layers.concatenate([u5, c1])
    c5 = layers.Conv2D(32, (3, 3), activation = 'relu', padding = 'same')(u5)
    c5 = layers.Conv2D(32, (3, 3), activation = 'relu', padding = 'same')(c5)

    # Output layer (sigmoid activation for pixel-wise binary classification)
    outputs = layers.Conv2D(1, (1, 1), activation = 'sigmoid')(c5)

    model = models.Model(inputs = [inputs], outputs = [outputs])
    return model

# Utilize Kaggle's dual GPU strategy for rapid model compilation
strategy = tf.distribute.MirroredStrategy()
print(f'\n🌐 Number of devices pooling for training: {strategy.num_replicas_in_sync}')

with strategy.scope():
    model = build_unet(input_shape=(256, 256, 2))
    # Compile with Adam optimizer and Binary Crossentropy Loss
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("🔥 U-Net Model Built and Compiled on Dual GPUs Successfully!")
model.summary()

### 🏋️‍♂️ Step 4: Execute Distributed Model Training
We define the training loop hyperparameters, configure early-stopping constraints to prevent overfitting, and trigger the execution across the paired Tesla T4 GPUs for 20 epochs.

In [ ]:
# Setting training hyperparameters
EPOCHS = 20
BATCH_SIZE = 16          # Dual GPU will divide per batch automatically

print("🚀 Engine Started! Training is launching on Dual GPUs...")

# Model training loop
history = model.fit(
    X_train, y_train,
    validation_data = (X_val, y_val),
    epochs = EPOCHS,
    batch_size = BATCH_SIZE,
    verbose = 1
)

print("\n🏆 BOOM! Training successfully complete!")

# Save the trained model file inside Kaggle working directory
model_save_path = '/kaggle/working/flood_unet_model.keras'
model.save(model_save_path)
print(f"💾 Model permanently save ho gaya hai yahan: {model_save_path}")

### 📊 Step 5: Visualizing Model Predictions on Validation Data
We pull a random unseen SAR temporal patch from the validation array, pass it through our trained U-Net model, and plot the side-by-side comparison of the Ground Truth vs. AI Prediction.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

# 1. Validation set (X_val) se ek random index select karo
idx = random.randint(0, len(X_val) - 1)

# 2. Random image patch aur uska asli mask uthao
sample_img = X_val[idx]
sample_mask = y_val[idx]

# 3. Safe Prediction: Expand dimension aur directly tensor call karo 
# Isse batch division zero nahi hoga dual GPU par
input_tensor = np.expand_dims(sample_img, axis=0)
pred_tensor = model(input_tensor, training=False)

# Convert prediction back to numpy array
pred_mask = pred_tensor.numpy()[0]

# 4. Thresholding (0.5 se upar wale pixels = Flood)
pred_mask_binary = (pred_mask > 0.5).astype(np.float32)

# 5. Side-by-Side Plotting
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

# Column 1: Pre-Flood SAR Image (Channel 0)
ax[0].imshow(sample_img[:, :, 0], cmap='gray')
ax[0].set_title(f"📡 SAR Pre-VV Patch (Index: {idx})", fontsize=12)
ax[0].axis('off')

# Column 2: Ground Truth (Asli Flood Mask)
ax[1].imshow(sample_mask.squeeze(), cmap='Blues')
ax[1].set_title("🗺️ Actual Ground Truth Mask", fontsize=12)
ax[1].axis('off')

# Column 3: AI Model Prediction (U-Net Output)
ax[2].imshow(pred_mask_binary.squeeze(), cmap='Blues')
ax[2].set_title("🧠 U-Net AI Predicted Flood", fontsize=12)
ax[2].axis('off')

plt.tight_layout()
plt.show()